# Eleição **Homer Simpson** x **Spock** — Apuração dinâmica por estado do Brasil

**Objetivo:** simular uma eleição entre **Homer Simpson** e **Spock** com **contagem dinâmica de votos por estado (UF)**, usando um pipeline de streaming real:

| Etapa | Tecnologia |
|-------|------------|
| Geração de votos | Produtor Python (urnas simuladas) |
| Transporte | **Apache Kafka** (tópico `votos`) |
| Processamento | **Apache Spark Structured Streaming** |
| Persistência | **MongoDB** (apuração acumulada) |
| Visualização | Consulta + gráfico do vencedor por estado |

> ⚠️ **Runtime → Factory reset runtime** antes de executar, para garantir uma VM limpa.

### Como a apuração é "dinâmica"?
O Spark mantém um **estado agregado** (`groupBy(estado, candidato).count()`) e, a cada micro-batch, emite apenas os grupos que mudaram (`outputMode("update")`). Esses totais são **atualizados (upsert)** no MongoDB, então a contagem por estado cresce em tempo real conforme os votos chegam pelo Kafka.


## 1. Instalar Java + Apache Kafka (modo KRaft, sem ZooKeeper)

O Kafka roda dentro da própria VM do Colab. Usamos o modo **KRaft** (Kafka 3.x) que dispensa o ZooKeeper.

In [ ]:
%%bash
set -e

echo "=== Instalando Java (necessário para Kafka e Spark) ==="
apt-get -qq update
DEBIAN_FRONTEND=noninteractive apt-get -qq install -y default-jdk > /dev/null

echo "=== Baixando Apache Kafka 3.7.1 ==="
cd /content
if [ ! -d kafka_2.12-3.7.1 ]; then
  wget -q https://archive.apache.org/dist/kafka/3.7.1/kafka_2.12-3.7.1.tgz
  tar -xzf kafka_2.12-3.7.1.tgz
fi

cd kafka_2.12-3.7.1
echo "=== Formatando storage (KRaft) e iniciando o broker ==="
KAFKA_CLUSTER_ID=$(bin/kafka-storage.sh random-uuid)
bin/kafka-storage.sh format -t "$KAFKA_CLUSTER_ID" -c config/kraft/server.properties > /dev/null 2>&1 || true
bin/kafka-server-start.sh -daemon config/kraft/server.properties
sleep 12

echo "=== Criando tópico 'votos' ==="
bin/kafka-topics.sh --create --topic votos \
  --bootstrap-server 127.0.0.1:9092 \
  --partitions 3 --replication-factor 1 2>/dev/null || echo "(tópico já existe)"

echo "=== Tópicos disponíveis ==="
bin/kafka-topics.sh --list --bootstrap-server 127.0.0.1:9092
echo "✅ Kafka rodando em 127.0.0.1:9092"


## 2. Instalar e iniciar o MongoDB local

Baixamos os binários do MongoDB Community e subimos o `mongod` na porta padrão `27017`.

In [ ]:
%%bash
set -e

echo "=== Baixando MongoDB Community 7.0 ==="
cd /content
MONGO_DIR=mongodb-linux-x86_64-ubuntu2204-7.0.14
if [ ! -d "$MONGO_DIR" ]; then
  wget -q https://fastdl.mongodb.org/linux/${MONGO_DIR}.tgz
  tar -xzf ${MONGO_DIR}.tgz
fi

mkdir -p /content/mongo_data
echo "=== Iniciando mongod em 127.0.0.1:27017 ==="
nohup /content/${MONGO_DIR}/bin/mongod \
  --dbpath /content/mongo_data \
  --bind_ip 127.0.0.1 --port 27017 \
  > /content/mongod.log 2>&1 &
sleep 8
tail -n 3 /content/mongod.log
echo "✅ MongoDB rodando em 127.0.0.1:27017"


## 3. Dependências Python + configuração

Fixamos o **PySpark 3.5.1** (Scala 2.12) para casar com o conector `spark-sql-kafka-0-10_2.12:3.5.1`.

In [ ]:
%pip -q install pyspark==3.5.1 kafka-python pymongo

import json
import time
import random
import threading
from datetime import datetime

from kafka import KafkaProducer
from pymongo import MongoClient

# ============================================================
# CONFIGURAÇÃO — TUDO LOCAL NA VM DO COLAB
# ============================================================
KAFKA_BOOTSTRAP = "127.0.0.1:9092"
KAFKA_TOPIC = "votos"

MONGO_URI = "mongodb://127.0.0.1:27017"
MONGO_DB = "eleicao"
MONGO_COLL = "apuracao"

CANDIDATOS = ["HOMER SIMPSON", "SPOCK"]

# 26 estados + Distrito Federal
ESTADOS = [
    "AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO", "MA",
    "MT", "MS", "MG", "PA", "PB", "PR", "PE", "PI", "RJ", "RN",
    "RS", "RO", "RR", "SC", "SP", "SE", "TO",
]

# "Peso" de cada candidato por estado (só para gerar uma eleição disputada e realista).
# Valor = probabilidade de o voto ser no Homer naquele estado.
PESO_HOMER = {uf: random.uniform(0.35, 0.65) for uf in ESTADOS}

DURACAO_PRODUTOR_SEG = 90     # tempo gerando votos
VOTOS_POR_LOTE = (20, 60)     # min/max de votos por rajada
INTERVALO_SEGUNDOS = 2

# Limpa apuração anterior
_mc = MongoClient(MONGO_URI)
_mc[MONGO_DB][MONGO_COLL].delete_many({})
_mc.close()

print("=" * 55)
print("  ELEIÇÃO: Homer Simpson  x  Spock")
print(f"  Kafka:   {KAFKA_BOOTSTRAP}  (tópico '{KAFKA_TOPIC}')")
print(f"  MongoDB: {MONGO_URI}  (db '{MONGO_DB}')")
print(f"  Estados: {len(ESTADOS)} UFs")
print("=" * 55)


## 4. Produtor — urnas eletrônicas enviando votos para o Kafka

Cada "urna" gera um voto (`candidato`, `estado`, `event_time`) e publica no tópico `votos`. Em produção, a fonte seria a rede de urnas / um gateway de eventos; aqui simulamos com o `KafkaProducer`.

In [ ]:
def gerar_voto() -> dict:
    uf = random.choice(ESTADOS)
    candidato = "HOMER SIMPSON" if random.random() < PESO_HOMER[uf] else "SPOCK"
    return {
        "id_urna": random.randint(1, 500_000),
        "estado": uf,
        "candidato": candidato,
        "event_time": datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S"),
    }


def produzir_votos(stop_event: threading.Event):
    producer = KafkaProducer(
        bootstrap_servers=KAFKA_BOOTSTRAP,
        key_serializer=lambda k: k.encode("utf-8"),
        value_serializer=lambda v: json.dumps(v).encode("utf-8"),
        acks="all",
    )
    inicio = time.time()
    total = 0
    lote = 0
    while not stop_event.is_set() and (time.time() - inicio) < DURACAO_PRODUTOR_SEG:
        lote += 1
        n = random.randint(*VOTOS_POR_LOTE)
        for _ in range(n):
            voto = gerar_voto()
            # chave = estado -> votos do mesmo estado caem na mesma partição
            producer.send(KAFKA_TOPIC, key=voto["estado"], value=voto)
        producer.flush()
        total += n
        print(f"🗳️  Lote {lote}: +{n} votos (acumulado enviado: {total})")
        time.sleep(INTERVALO_SEGUNDOS)
    producer.flush()
    producer.close()
    print(f"🏁 Produtor encerrado. Total enviado: {total} votos.")


# Amostra
print("Amostra de voto:", json.dumps(gerar_voto(), indent=2, ensure_ascii=False))


## 5. Spark Session com o conector Kafka

In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

SPARK_VER = pyspark.__version__  # 3.5.1
KAFKA_PKG = f"org.apache.spark:spark-sql-kafka-0-10_2.12:{SPARK_VER}"

spark = (
    SparkSession.builder
    .appName("EleicaoHomerSpockKafkaMongo")
    .config("spark.jars.packages", KAFKA_PKG)
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} | pacote Kafka: {KAFKA_PKG}")


## 6. Structured Streaming: Kafka → agregação por estado → MongoDB

1. Lemos o stream do tópico `votos`.
2. Parseamos o JSON.
3. Agrupamos por **estado + candidato** e contamos (`count()`) — esse é o **estado acumulado**.
4. Com `outputMode("update")`, cada micro-batch traz os grupos que mudaram.
5. No `foreachBatch`, fazemos **upsert** no MongoDB, mantendo a contagem sempre atualizada.

In [ ]:
schema = StructType([
    StructField("id_urna", IntegerType(), True),
    StructField("estado", StringType(), False),
    StructField("candidato", StringType(), False),
    StructField("event_time", StringType(), True),
])

votos_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
    .option("subscribe", KAFKA_TOPIC)
    .option("startingOffsets", "earliest")
    .load()
)

votos_df = (
    votos_raw
    .select(from_json(col("value").cast("string"), schema).alias("v"))
    .select("v.*")
)

# Estado acumulado: total de votos por (estado, candidato)
apuracao_df = votos_df.groupBy("estado", "candidato").count()


def grava_mongo(batch_df, batch_id: int):
    # Upsert dos totais acumulados no MongoDB
    linhas = batch_df.collect()
    if not linhas:
        return

    client = MongoClient(MONGO_URI)
    coll = client[MONGO_DB][MONGO_COLL]
    for r in linhas:
        _id = f"{r['estado']}|{r['candidato']}"
        coll.update_one(
            {"_id": _id},
            {"$set": {
                "estado": r["estado"],
                "candidato": r["candidato"],
                "votos": int(r["count"]),
                "atualizado_em": datetime.utcnow(),
            }},
            upsert=True,
        )
    client.close()
    print(f"Batch {batch_id}: {len(linhas)} par(es) (estado, candidato) atualizados no MongoDB.")


query = (
    apuracao_df.writeStream
    .outputMode("update")
    .foreachBatch(grava_mongo)
    .option("checkpointLocation", "/content/checkpoint_eleicao")
    .trigger(processingTime="8 seconds")
    .start()
)

print(f"Streaming query id: {query.id}")


## 7. Executar as urnas + acompanhar a apuração ao vivo

O produtor roda em background enviando votos para o Kafka. O Spark consome, agrega e atualiza o MongoDB. Vamos imprimir a **apuração parcial** a cada poucos segundos.

In [ ]:
def placar_atual():
    client = MongoClient(MONGO_URI)
    docs = list(client[MONGO_DB][MONGO_COLL].find({}))
    client.close()
    total = {c: 0 for c in CANDIDATOS}
    for d in docs:
        total[d["candidato"]] = total.get(d["candidato"], 0) + d["votos"]
    return total


stop_event = threading.Event()
produtor = threading.Thread(target=produzir_votos, args=(stop_event,), daemon=True)
produtor.start()

try:
    tempo_total = DURACAO_PRODUTOR_SEG + 25
    fim = time.time() + tempo_total
    while time.time() < fim:
        time.sleep(10)
        t = placar_atual()
        h, s = t.get("HOMER SIMPSON", 0), t.get("SPOCK", 0)
        lider = "HOMER" if h > s else ("SPOCK" if s > h else "EMPATE")
        print(f"📊 Parcial nacional → Homer: {h}  |  Spock: {s}  →  líder: {lider}")
finally:
    stop_event.set()
    produtor.join(timeout=10)
    time.sleep(8)          # deixa o Spark drenar os últimos votos
    query.stop()
    print("✅ Streaming finalizado.")


## 8. Resultado por estado (quem ganhou em cada UF)

Consulta de agregação no MongoDB para decidir o **vencedor por estado** e o **placar nacional**.

In [ ]:
import pandas as pd

client = MongoClient(MONGO_URI)
docs = list(client[MONGO_DB][MONGO_COLL].find({}))
client.close()

df = pd.DataFrame(docs)
if df.empty:
    print("Nenhum voto apurado. Rode as células anteriores novamente.")
else:
    pivot = (
        df.pivot_table(index="estado", columns="candidato",
                       values="votos", aggfunc="sum", fill_value=0)
        .reindex(columns=CANDIDATOS, fill_value=0)
    )
    pivot["VENCEDOR"] = pivot.apply(
        lambda r: "HOMER SIMPSON" if r["HOMER SIMPSON"] > r["SPOCK"]
        else ("SPOCK" if r["SPOCK"] > r["HOMER SIMPSON"] else "EMPATE"),
        axis=1,
    )
    pivot = pivot.sort_index()
    display(pivot)


## 9. Vencedor da eleição (por estados vencidos e por votos totais)

In [ ]:
if not df.empty:
    estados_homer = int((pivot["VENCEDOR"] == "HOMER SIMPSON").sum())
    estados_spock = int((pivot["VENCEDOR"] == "SPOCK").sum())
    empates = int((pivot["VENCEDOR"] == "EMPATE").sum())

    votos_homer = int(pivot["HOMER SIMPSON"].sum())
    votos_spock = int(pivot["SPOCK"].sum())

    print("=" * 50)
    print("  APURAÇÃO FINAL")
    print("=" * 50)
    print(f"  Estados vencidos → Homer: {estados_homer} | Spock: {estados_spock} | Empates: {empates}")
    print(f"  Votos totais     → Homer: {votos_homer} | Spock: {votos_spock}")
    print("-" * 50)

    if votos_homer > votos_spock:
        campeao = "🍩 HOMER SIMPSON venceu no voto popular!"
    elif votos_spock > votos_homer:
        campeao = "🖖 SPOCK venceu no voto popular! (Live long and prosper)"
    else:
        campeao = "🤝 EMPATE no voto popular!"
    print(f"  {campeao}")
    print("=" * 50)


## 10. Visualização — votos por candidato e mapa de estados vencidos

In [ ]:
import matplotlib.pyplot as plt

if not df.empty:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Barras: total nacional por candidato
    ax1.bar(["Homer Simpson", "Spock"], [votos_homer, votos_spock],
            color=["#FFD521", "#1f3b73"])
    ax1.set_title("Votos totais por candidato")
    ax1.set_ylabel("Votos")
    for i, v in enumerate([votos_homer, votos_spock]):
        ax1.text(i, v, str(v), ha="center", va="bottom", fontweight="bold")

    # Barras por estado (empilhado horizontal)
    pv = pivot.sort_values("HOMER SIMPSON")
    ax2.barh(pv.index, pv["HOMER SIMPSON"], color="#FFD521", label="Homer")
    ax2.barh(pv.index, pv["SPOCK"], left=pv["HOMER SIMPSON"],
             color="#1f3b73", label="Spock")
    ax2.set_title("Votos por estado (UF)")
    ax2.set_xlabel("Votos")
    ax2.legend()
    ax2.tick_params(axis="y", labelsize=7)

    plt.tight_layout()
    plt.show()


## 11. (Opcional) Encerrar Spark

In [ ]:
try:
    query.stop()
except Exception:
    pass
spark.stop()
print("🛑 Spark encerrado.")


## 12. Deploy (fora do Colab)

Para rodar este pipeline em produção/servidor, use o **Docker Compose** e os scripts em [`deploy/`](https://github.com/naubergois/SparkSteammingAulas/tree/main/deploy):

```bash
# 1. Sobe Kafka + MongoDB
cd deploy
docker compose up -d

# 2. Instala dependências e roda o produtor de votos
pip install -r requirements.txt
python producer.py

# 3. Em outro terminal, roda o job Spark Streaming (Kafka -> MongoDB)
spark-submit \
  --packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1 \
  spark_streaming_job.py
```

- **Kafka**: `localhost:9092` (tópico `votos`)
- **MongoDB**: `localhost:27017` (db `eleicao`, coleção `apuracao`)
- **Mongo Express** (UI opcional): http://localhost:8081

Os arquivos `docker-compose.yml`, `producer.py`, `spark_streaming_job.py` e `requirements.txt` estão na pasta `deploy/` deste repositório.
